# FPR Experiment for Adaptive CoRT-SI across different nT
This notebook tests the false positive rate (FPR) of Adaptive CoRT-SI across varying target sample sizes (`nT`). It supports resumability, meaning if the execution is interrupted, it can pick up exactly where it left off by reading the saved `.npz` files.

In [25]:
from pathlib import Path
import sys

import numpy as np

cwd = Path.cwd().resolve()
candidates = [cwd, cwd.parent]
REPO_ROOT = next((candidate for candidate in candidates if (candidate / "cort_si").exists()), None)
if REPO_ROOT is None:
    raise RuntimeError("Could not locate the repo root containing the cort_si package.")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from cort_si import SI_parallel_randj, SI_randj, gen_data

RESULTS_DIR = REPO_ROOT / "run_experiment" / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

In [26]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
p = 300
nS = 100
num_info_aux = 3
num_uninfo_aux = 2
K = num_info_aux + num_uninfo_aux
lambda_sel = 0.8
lambda0 = 0.8
T = 5
nT_list = [50, 75, 100]

num_runs = 1000
base_seed = 0
alpha = 0.05
true_beta_scale = 1.0
rho = 0.0
sigma_noise = 1.0
source_shift_sd = 0.3
covariate_shift = False
save_interval = 1

print(f"num_runs = {num_runs}")
print(f"fixed alpha = {alpha}")
print(f"Null model true_beta scale = {true_beta_scale}")
print(f"K = {K}, p = {p}, nS = {nS}")
print(f"Testing nT values: {nT_list}")

num_runs = 1000
fixed alpha = 0.05
Null model true_beta scale = 0.0
K = 5, p = 300, nS = 100
Testing nT values: [50, 75, 100]


In [ ]:
def generate_null_instance(nT_val, iteration_seed):
    return gen_data.generate_data(
        p=p,
        nS=nS,
        nT=nT_val,
        K=K,
        true_beta=true_beta_scale,
        rho=rho,
        sigma_noise=sigma_noise,
        source_shift_sd=source_shift_sd,
        covariate_shift=covariate_shift,
        seed=iteration_seed,
        num_info_aux=num_info_aux,
        num_uninfo_aux= num_uninfo_aux
    )

def get_save_path(p_val, nS_val, nT_val, K, lambda_sel, lambda0):
    return RESULTS_DIR / f"tpr_p{p_val}_nS{nS_val}_nT{nT_val}_K{K}_lamsel{lambda_sel}_lam0{lambda0}.npz"

import numpy as np

def load_progress(filepath):
    if filepath.exists():
        data = np.load(filepath)
        # Hỗ trợ cả file cũ (chỉ có null_pvalues) và file mới (có thêm j_values)
        if 'j_values' in data and 'null_pvalues' in data:
            return list(data['j_values']), list(data['null_pvalues'])
        elif 'null_pvalues' in data:
            return [], list(data['null_pvalues'])
    return [], []

def save_progress(filepath, j_values, null_pvalues, p_val, nS_val, nT_val, num_runs_val, alpha_val):
    np.savez(
        filepath,
        j_values=np.asarray(j_values, dtype=int),
        null_pvalues=np.asarray(null_pvalues, dtype=float),
        p=np.asarray([p_val], dtype=int),
        nS=np.asarray([nS_val], dtype=int),
        nT=np.asarray([nT_val], dtype=int),
        num_runs=np.asarray([num_runs_val], dtype=int),
        alpha=np.asarray([alpha_val], dtype=float),
    )


In [ ]:
def run_resumable_fpr_experiment(nT_val, total_runs, seed_base, save_intvl=10):
    filepath = get_save_path(p, nS, nT_val, K, lambda_sel, lambda0)
    j_values, null_pvalues = load_progress(filepath)
    start_iteration = len(null_pvalues)
    
    if start_iteration >= total_runs:
        print(f"Experiment for nT={nT_val} already completed ({total_runs} runs).")
        return j_values, null_pvalues
    
    print(f"Starting/Resuming experiment for nT={nT_val} from run {start_iteration}/{total_runs}...")
    cnt = 0
    sum = 0
    for iteration in range(start_iteration, total_runs):
        data_seed = seed_base + iteration
        XS_list, YS_list, X0, Y0, _, SigmaS_list, Sigma0, _ = generate_null_instance(nT_val, data_seed)
        lambdak_list = [lambda0] * len(XS_list)

        # SI_parallel_randj giờ trả về (j, p_sel) hoặc None
        result = SI_parallel_randj(
            X0,
            Y0,
            XS_list,
            YS_list,
            lambda_sel=lambda_sel,
            lambda0=lambda0,
            lambdak_list=lambdak_list,
            SigmaS_list=SigmaS_list,
            Sigma0=Sigma0,
            T=T,
            verbose=False,
        )

        if result is not None:
            j_val, p_sel = result
            if p_sel is not None and np.isfinite(p_sel):
                if j <= 5:
                    if p_sel <= alpha:
                        cnt += 1
                    sum += 1
                j_values.append(int(j_val))
                null_pvalues.append(float(p_sel))

        if (iteration + 1) % save_intvl == 0 or (iteration + 1) == total_runs:
            save_progress(filepath, j_values, null_pvalues, p, nS, nT_val, total_runs, alpha)
            print(f"  [nT={nT_val}] Saved progress: {iteration + 1}/{total_runs} runs completed.")
            if sum > 0:
                print(f"Empirical FPR @ alpha={alpha}: {(cnt / sum):.3f} | total run : {sum}")

    return j_values, null_pvalues


In [30]:
for current_nT in nT_list:
    run_resumable_fpr_experiment(current_nT, num_runs, base_seed, save_intvl=save_interval)

Starting/Resuming experiment for nT=50 from run 0/1000...
  [nT=50] Saved progress: 1/1000 runs completed.
Empirical FPR @ alpha=0.05: 0.000 | total run : 1
  [nT=50] Saved progress: 2/1000 runs completed.
Empirical FPR @ alpha=0.05: 0.000 | total run : 2
  [nT=50] Saved progress: 3/1000 runs completed.
Empirical FPR @ alpha=0.05: 0.000 | total run : 2
  [nT=50] Saved progress: 4/1000 runs completed.
Empirical FPR @ alpha=0.05: 0.000 | total run : 2
  [nT=50] Saved progress: 5/1000 runs completed.
Empirical FPR @ alpha=0.05: 0.000 | total run : 2
  [nT=50] Saved progress: 6/1000 runs completed.
Empirical FPR @ alpha=0.05: 0.000 | total run : 2
  [nT=50] Saved progress: 7/1000 runs completed.
Empirical FPR @ alpha=0.05: 0.000 | total run : 2
  [nT=50] Saved progress: 8/1000 runs completed.
Empirical FPR @ alpha=0.05: 0.000 | total run : 3
  [nT=50] Saved progress: 9/1000 runs completed.
Empirical FPR @ alpha=0.05: 0.000 | total run : 4
  [nT=50] Saved progress: 10/1000 runs completed.
E

KeyboardInterrupt: 

In [ ]:
print("=== FPR Experiment Summary ===")
for current_nT in nT_list:
    filepath = get_save_path(p, nS, current_nT, K, lambda_sel, lambda0)
    j_values, pvals = load_progress(filepath)
    
    if not pvals:
        print(f"nT={current_nT}: No data found.")
        continue
    
    pvals_arr = np.asarray(pvals)
    # Tính FPR: tỉ lệ số lần p-value <= alpha
    rejection_rate = (pvals_arr <= alpha).mean()
    print(f"nT={current_nT}: {len(pvals)} valid runs | Empirical FPR @ alpha={alpha}: {rejection_rate:.3f}")

=== FPR Experiment Summary ===
nT=50: 222 valid runs | Empirical FPR @ alpha=0.05: 0.302
nT=75: No data found.
nT=100: No data found.
